## Libraries

In [0]:
from pyspark.sql.functions import col, avg, when
from pyspark.sql import DataFrame
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import concat_ws

## Load data

In [0]:
clean_data = spark.table('workspace.telco.bronze_data')

In [0]:
display(clean_data.limit(5))

## Features

I will create features from indexed_data grouping by different categorical columns. 

### Numerical features

In [0]:
categorical_cols = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod"]

In [0]:
def feat_related_charges(df: DataFrame, column:str):
    # Precompute tenure-safe & ratios only once
    df = df.withColumn("tenure_safe", when(col("tenure") == 0, None).otherwise(col("tenure"))) \
           .withColumn("monthly_charge_per_tenure", col("MonthlyCharges") / col("tenure_safe")) \
           .withColumn("total_charge_per_tenure", col("TotalCharges") / col("tenure_safe"))

    # ONE aggregation instead of 5
    agg_df = df.groupBy(column).agg(
        avg("tenure").alias(f"avg_tenure_by_{column}"),
        avg("MonthlyCharges").alias(f"avg_monthly_charge_by_{column}"),
        avg("TotalCharges").alias(f"avg_total_charge_by_{column}"),
        avg("monthly_charge_per_tenure").alias(f"avg_monthly_charge_per_tenure_by_{column}"),
        avg("total_charge_per_tenure").alias(f"avg_total_charge_per_tenure_by_{column}")
    )

    # ONE join instead of 5
    df = df.join(agg_df, on=column, how="left")
    return df

for column in categorical_cols:
    clean_data = feat_related_charges(clean_data, column)

## Combining categorical features

In [0]:
categorical_cols_2_concatenate = ["gender", "SeniorCitizen", "Partner", "Dependents", "PhoneService", "InternetService", "TechSupport", "StreamingTV", "StreamingMovies", "Contract"]

In [0]:
def concatenate_cat_cols(df, column1, column2):
    return df.withColumn(column1 + "_" + column2, concat_ws("_", col(column1), col(column2)))

In [0]:
import gc 
gc.collect()

In [0]:
for column in categorical_cols_2_concatenate:
    for column2 in categorical_cols_2_concatenate:
        if column != column2:
            clean_data = concatenate_cat_cols(clean_data, column, column2)
            

In [0]:
# MonthlyCharges/tenure by 